# ViSEF — Đánh giá tổng hợp có thể tái lập

Notebook này hợp nhất kiểm tra dữ liệu, benchmark đa seed, chọn checkpoint bằng validation, đánh giá test, bảng mean ± std/error bars, sweep epsilon, MIA và đối chiếu accountant. Chạy tuần tự trên Colab có GPU; có thể resume sau khi ngắt.

Mặc định 3 seed × 4 cấu hình. Mọi cấu hình dùng cùng partition và `linear` (chỉ classifier cuối, khoảng 4 nghìn tham số) để kiểm tra giả thuyết giảm số chiều giúp DP-SGD. Accuracy phải được đo; không giả định trước mức cải thiện.

In [ ]:
# 1) Cai dat moi truong
!pip install -q torch torchvision numpy Pillow matplotlib
!pip install -q opacus   # de --verify_dp doi chieu accountant
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2) Tai va chuan hoa du lieu
%cd /content
!sed -i 's/\r$//' src/prepare_data.sh
!bash src/prepare_data.sh

In [ ]:
# 3) Kiem tra cac file code da co
import os
required = ['src/common.py','src/client.py','src/server.py','src/partition.py','src/evaluate.py',
            'src/privacy.py','src/run_simulation.py','src/prepare_data.sh',
            'src/dedup_check.py','src/centralized.py']
missing = [f for f in required if not os.path.exists(f)]
assert not missing, f'Thieu file: {missing}'
print('OK - day du file:', required)

## Bước 1 — Dedup check (Priority 1, item 5)

Bộ Brain Tumor MRI Dataset (Kaggle) đã có báo cáo trùng lặp ảnh giữa Training và Testing. Nếu bỏ qua, accuracy có thể bị thổi phồng. Chạy kiểm tra trước; nếu phát hiện nhiều, chạy với `--remove` rồi **phân hoạch lại**.

In [ ]:
!python src/dedup_check.py --data_root ./brain-tumor-mri-dataset --threshold 0

# Neu co nhieu cap trung lap, bo comment dong duoi de xoa anh Training trung voi Testing:
# !python src/dedup_check.py --data_root ./brain-tumor-mri-dataset --threshold 0 --remove
# Sau khi --remove: phai phan hoach lai (REPARTITION=1 bash src/prepare_data.sh)

## Bước 2 — Cấu hình multi-seed

Đặt `QUICK_MODE=True` nếu muốn chạy thử nhanh (5 rounds, 1 seed). Bản nộp nên dùng cấu hình đầy đủ (3 seed × 20 rounds).

In [ ]:
QUICK_MODE = False
SEEDS = [42, 123, 456] if not QUICK_MODE else [42]
ROUNDS = 20 if not QUICK_MODE else 5
EPOCHS, BATCH, LR = 3, 32, 1e-4
DOM_FRAC = 0.7
TRAIN_MODE = 'linear'  # chỉ Linear cuối; cùng train_mode cho cả 4 cấu hình
CONFIGS = {
    'fedavg': dict(mu=0.0, dp=False),
    'fedprox': dict(mu=0.01, dp=False),
    'fedprox_dp': dict(mu=0.01, dp=True),
    'fedprox_dp_autoclip': dict(mu=0.01, dp=True, auto_clip=True),
}
print(f'Seeds={SEEDS} | rounds={ROUNDS} | train_mode={TRAIN_MODE} | configs={list(CONFIGS)}')


## Bước 3 — Chạy 12 lần (có resume)

Mỗi lần: `run_simulation --skip_eval` → `evaluate --split val --all_rounds` → chọn round tốt nhất theo val Macro-F1 → `evaluate --split test --server_rounds <best>`.

Nếu cell bị ngắt, **chạy lại cell này** — run đã hoàn tất sẽ được bỏ qua.

In [ ]:
import subprocess, sys, json, time
from pathlib import Path

def sh(cmd):
    print('\n$', ' '.join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True)

def pick_best_round(run_dir):
    val = json.load(open(Path(run_dir) / 'eval' / 'eval_val.json'))['results']
    srv = [r for r in val if r['kind'] == 'server' and r['round'] > 0]
    if not srv:
        return None
    return max(srv, key=lambda r: r['macro_f1'])['round']

t_start = time.time()
for seed in SEEDS:
    for name, cfg in CONFIGS.items():
        tag = f'p4_linear_{name}_seed{seed}'
        run_dir = Path('runs') / tag
        test_json = run_dir / 'eval' / 'eval_test.json'
        if test_json.exists():
            print(f'[SKIP] {tag} da hoan tat')
            continue
        print(f'\n========== {tag} ==========')
        cmd = [sys.executable, 'src/run_simulation.py',
               '--data_root', './brain-tumor-mri-dataset',
               '--tag', tag,
               '--partition_dir', 'partitions',
               '--rounds', str(ROUNDS), '--epochs', str(EPOCHS),
               '--batch_size', str(BATCH), '--lr', str(LR),
               '--train_mode', TRAIN_MODE,
               '--mu', str(cfg['mu']),
               '--dominant_frac', str(DOM_FRAC),
               '--seed', str(seed), '--skip_eval']
        if cfg['dp']:
            cmd += ['--dp', '--noise_multiplier', '1.0',
                    '--max_grad_norm', '1.0', '--verify_dp']
            if cfg.get('auto_clip'):
                cmd += ['--auto_clip']
        if seed == SEEDS[0]:
            cmd.append('--repartition')
        sh(cmd)
        # validation moi round -> chon best round
        sh([sys.executable, 'src/evaluate.py', '--run_dir', run_dir,
            '--split', 'val', '--partition_dir', 'partitions',
            '--all_rounds', '--device', 'auto', '--no_plots'])
        best = pick_best_round(run_dir)
        print(f'[Pick] {tag}: best round = {best}')
        # test CHI tren best round
        sh([sys.executable, 'src/evaluate.py', '--run_dir', run_dir,
            '--split', 'test', '--data_root', './brain-tumor-mri-dataset',
            '--server_rounds', str(best), '--client_rounds', '',
            '--device', 'auto', '--no_plots'])

print(f'\nTong thoi gian: {(time.time()-t_start)/60:.1f} phut')

## Bước 4 — Aggregate mean ± std

Đọc `eval_test.json` của 12 run, tính mean và std trên 3 seed cho từng cấu hình. Đây là bảng sẽ đưa vào báo cáo.

In [ ]:
import pandas as pd, numpy as np

rows = []
for seed in SEEDS:
    for name in CONFIGS:
        tag = f'p4_linear_{name}_seed{seed}'
        test_json = Path('runs') / tag / 'eval' / 'eval_test.json'
        if not test_json.exists():
            print(f'[MISS] {tag}'); continue
        t = json.load(open(test_json))['results'][0]
        val = json.load(open(Path('runs')/tag/'eval'/'eval_val.json'))['results']
        srv = [r for r in val if r['kind'] == 'server' and r['round'] > 0]
        best = max(srv, key=lambda r: r['macro_f1'])
        rows.append({
            'config': name, 'seed': seed, 'best_round': best['round'],
            'val_acc': best['accuracy'], 'val_f1': best['macro_f1'],
            'test_acc': t['accuracy'], 'test_f1': t['macro_f1'],
            'test_balacc': t['balanced_accuracy'],
            **{f'recall_{i}': r for i, r in enumerate(t['per_class_recall'])},
        })

df = pd.DataFrame(rows)
if len(SEEDS) < 2:
    raise RuntimeError('Quick mode is only a smoke run; use at least two seeds for mean ± std and error bars.')
counts = df.groupby('config')['seed'].nunique().reindex(CONFIGS, fill_value=0)
if (counts < len(SEEDS)).any():
    raise RuntimeError(f'Chua du seed de tinh mean ± std; can {len(SEEDS)} moi cau hinh, hien co: {counts.to_dict()}')
df.to_csv('runs/linear_multiseed.csv', index=False)
df

In [ ]:
summary = df.groupby('config').agg({
    'test_acc':   ['mean','std'],
    'test_f1':    ['mean','std'],
    'test_balacc':['mean','std'],
}).round(4)
print('=== Test metrics: mean ± std (3 seeds) ===')
print(summary.to_string())
summary.to_csv('runs/linear_summary.csv')

print('\n=== Recall theo lop (mean ± std) ===')
for i, cname in enumerate(['glioma','meningioma','notumor','pituitary']):
    sub = df.groupby('config')[f'recall_{i}'].agg(['mean','std']).round(3)
    print(f'\n{cname}:'); print(sub.to_string())

## Bước 5 — Biểu đồ error bars

Ba biểu đồ: accuracy, macro-F1, balanced accuracy — mean ± std trên 3 seed. Đây là hình chính cho báo cáo (thay cho bảng 1 lần chạy cũ).

In [ ]:
import matplotlib.pyplot as plt

configs = list(CONFIGS)
colors  = {'fedavg':'tab:blue','fedprox':'tab:orange','fedprox_dp':'tab:red','fedprox_dp_autoclip':'tab:purple'}
titles  = ['Accuracy','Macro-F1','Balanced Acc.']
cols    = ['test_acc','test_f1','test_balacc']

fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, col, title in zip(axes, cols, titles):
    means = [df[df['config']==c][col].mean()*100 for c in configs]
    stds  = [df[df['config']==c][col].std()*100  for c in configs]
    bars = ax.bar(configs, means, yerr=stds, capsize=8,
                  color=[colors[c] for c in configs], edgecolor='black')
    for b, m, s in zip(bars, means, stds):
        ax.text(b.get_x()+b.get_width()/2, m+s+1.0, f'{m:.1f}±{s:.1f}',
                ha='center', fontsize=9)
    ax.set_title(title); ax.set_ylabel('%'); ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('Priority 1 - Multi-seed benchmark (mean ± std, 3 seeds)', fontweight='bold')
fig.tight_layout()
fig.savefig('runs/linear_summary.png', dpi=150)
plt.show()

In [ ]:
# Recall theo lop: so sanh 3 cau hinh
classes = ['glioma','meningioma','notumor','pituitary']
fig, ax = plt.subplots(figsize=(9,4.5))
x = np.arange(len(classes)); w = 0.8 / len(configs)
for i, c in enumerate(configs):
    sub = df[df['config']==c]
    m = [sub[f'recall_{j}'].mean()*100 for j in range(4)]
    s = [sub[f'recall_{j}'].std()*100  for j in range(4)]
    ax.bar(x + i*w, m, w, yerr=s, capsize=3, label=c, color=colors[c])
ax.set_xticks(x + w*(len(configs)-1)/2); ax.set_xticklabels(classes)
ax.set_ylabel('Recall (%)'); ax.set_title('Per-class recall (mean ± std)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); fig.savefig('runs/linear_recall.png', dpi=150)
plt.show()

## Bước 6 (tuỳ chọn) — Baseline centralized

Chạy centralized (60 epoch ≈ 20 vòng × 3 epoch, cùng ngân sách update) để trả lời câu hỏi *"FL mất bao nhiêu so với huấn luyện tập trung?"*. **Không bắt buộc** cho Priority 1 nhưng rất nên có trong báo cáo.

In [ ]:
for seed in SEEDS:
    tag = f'p4_linear_centralized_seed{seed}'
    run_dir = Path('runs') / tag
    if (run_dir/'global'/'global_round_00.pt').exists():
        print(f'[SKIP] {tag}'); continue
    sh([sys.executable, 'src/centralized.py',
        '--data_root', './brain-tumor-mri-dataset',
        '--run_dir', run_dir,
        '--epochs', str(ROUNDS*EPOCHS),
        '--batch_size', str(BATCH), '--lr', str(LR),
        '--train_mode', TRAIN_MODE, '--seed', str(seed)])
    sh([sys.executable, 'src/evaluate.py', '--run_dir', run_dir,
        '--split', 'test', '--data_root', './brain-tumor-mri-dataset',
        '--server_rounds', '0', '--client_rounds', '',
        '--device', 'auto', '--no_plots'])

## Bước 7 — Xuất file cho báo cáo

Tải xuống `runs/linear_summary.csv`, `runs/linear_multiseed.csv`, `runs/linear_summary.png`, `runs/linear_recall.png` để chèn vào LaTeX.

## Phần B — Sweep σ → ε mục tiêu 1, 2, 3

Chỉ 1 seed (42), dùng clipping cố định C=1. `--target_epsilon` là ngân sách tích lũy cho toàn bộ số vòng đã định; client chọn σ theo tổng số bước còn lại và RDP đã tiêu. AutoClip được loại khỏi sweep vì ngưỡng clipping hiện hiệu chỉnh từ dữ liệu riêng mà chưa được tính privacy cost. Kết quả vẫn chỉ là khảo sát một seed.

In [ ]:
SWEEP_SEED   = 42
base_test = Path(f'runs/p4_linear_fedprox_seed{SWEEP_SEED}')/'eval'/'eval_test.json'
assert base_test.exists(), 'Run the multi-seed section above first to create the linear FedProx baseline.'
TARGET_EPS   = [1.0, 2.0, 3.0]
sweep_rows = []
for eps_t in TARGET_EPS:
    tag = f'p2_sweep_eps{eps_t:.1f}_seed{SWEEP_SEED}'
    run_dir = Path('runs')/tag
    if not (run_dir/'eval'/'eval_test.json').exists():
        print(f'\n========== {tag} ==========')
        cmd = [sys.executable,'src/run_simulation.py',
               '--data_root','./brain-tumor-mri-dataset','--tag',tag,
               '--partition_dir','partitions',
               '--rounds',str(ROUNDS),'--epochs',str(EPOCHS),
               '--batch_size',str(BATCH),'--lr',str(LR),
               '--train_mode',TRAIN_MODE,'--mu','0.01',
               '--dominant_frac',str(DOM_FRAC),
               '--seed',str(SWEEP_SEED),'--skip_eval',
               '--dp','--target_epsilon',str(eps_t),
               '--max_grad_norm','1.0','--verify_dp']
        sh(cmd)
        sh([sys.executable,'src/evaluate.py','--run_dir',run_dir,
            '--split','val','--partition_dir','partitions',
            '--all_rounds','--device','auto','--no_plots'])
        best = pick_best_round(run_dir)
        sh([sys.executable,'src/evaluate.py','--run_dir',run_dir,
            '--split','test','--data_root','./brain-tumor-mri-dataset',
            '--server_rounds',str(best),'--client_rounds','',
            '--device','auto','--no_plots'])
    # doc epsilon thuc te tu log aggregate cuoi cung
    logs = sorted((run_dir/'logs').glob('aggregate_round_*.json'))
    eps_real = None
    if logs:
        lg = json.load(open(logs[-1]))
        eps_vals = [v for v in lg.get('dp_epsilon',{}).values() if v and v>0]
        if eps_vals: eps_real = max(eps_vals)
    tj = run_dir/'eval'/'eval_test.json'
    if tj.exists():
        t = json.load(open(tj))['results'][0]
        val = json.load(open(run_dir/'eval'/'eval_val.json'))['results']
        srv = [r for r in val if r['kind']=='server' and r['round']>0]
        best_m = max(srv, key=lambda r:r['macro_f1'])
        sweep_rows.append({'target_eps':eps_t,'real_eps':eps_real,
                           'best_round':best_m['round'],
                           'val_f1':best_m['macro_f1'],
                           'test_acc':t['accuracy'],'test_f1':t['macro_f1']})

df_sweep = pd.DataFrame(sweep_rows)
# them baseline khong DP
base_tag = f'p4_linear_fedprox_seed{SWEEP_SEED}'
bt = Path('runs')/base_tag/'eval'/'eval_test.json'
if bt.exists():
    b = json.load(open(bt))['results'][0]
    df_sweep = pd.concat([pd.DataFrame([{'target_eps':float('inf'),'real_eps':0.0,
        'best_round':None,'val_f1':None,
        'test_acc':b['accuracy'],'test_f1':b['macro_f1']}]), df_sweep],
        ignore_index=True)
df_sweep.to_csv('runs/p2_sweep.csv', index=False)
print('\n=== Sweep σ → ε ===')
print(df_sweep.to_string())

In [ ]:
# Ve duong cong accuracy vs epsilon
fig, ax = plt.subplots(figsize=(7,4.5))
d = df_sweep.dropna(subset=['real_eps']).sort_values('real_eps')
ax.plot(d['real_eps'], d['test_acc']*100, 'o-', label='Test accuracy', lw=2)
ax.plot(d['real_eps'], d['test_f1']*100, 's--', label='Test macro-F1', lw=2)
for _, r in d.iterrows():
    x = r['real_eps']
    ax.annotate(f"ε={x:.2f}",
                (x, r['test_acc']*100), textcoords='offset points',
                xytext=(0,8), ha='center', fontsize=8)
ax.set_xlabel('Epsilon (ε, δ=1e-5)'); ax.set_ylabel('%')
ax.set_title('Trade-off accuracy – privacy (FedProx + DP-SGD, seed 42)')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig('runs/p2_sweep_curve.png', dpi=150)
plt.show()

## Phần C — Membership Inference Attack

Đo AUC trên checkpoint best của 3 cấu hình. Kỳ vọng: `fedavg`/`fedprox` (không DP) có AUC cao (rò rỉ), `fedprox_dp` có AUC gần 0,5 hơn (giảm rò rỉ).

In [ ]:
mia_rows = []
for seed in SEEDS:
    for name, prefix in [('fedavg','p4_linear_fedavg'),
                         ('fedprox','p4_linear_fedprox'),
                         ('fedprox_dp','p4_linear_fedprox_dp')]:
        tag = f'{prefix}_seed{seed}'
        run_dir = Path('runs')/tag
        if not (run_dir/'eval'/'eval_val.json').exists(): continue
        best = pick_best_round(run_dir)
        out = run_dir/'mia.json'
        # Always recompute: older files may include 400 non-member training images.
        print(f'\n[MIA] {tag} @ round {best}')
        sh([sys.executable,'src/mia_eval.py',
            '--data_root','./brain-tumor-mri-dataset',
            '--run_dir',str(run_dir),'--round',str(best),
            '--out',str(out),'--device','auto','--partition_dir','partitions'])
        r = json.load(open(out))
        r['config']=name; r['seed']=seed
        mia_rows.append(r)

df_mia = pd.DataFrame(mia_rows)
if len(SEEDS) < 2:
    raise RuntimeError('Quick mode is only a smoke run; use at least two seeds for MIA mean ± std.')
mia_counts = df_mia.groupby('config')['seed'].nunique().reindex(['fedavg','fedprox','fedprox_dp'], fill_value=0)
if (mia_counts < len(SEEDS)).any():
    raise RuntimeError(f'Incomplete MIA runs; expected {len(SEEDS)} seeds/config, got {mia_counts.to_dict()}')
df_mia.to_csv('runs/p3_mia.csv', index=False)
print('\n=== MIA AUC (mean ± std tren cac seed) ===')
print(df_mia.groupby('config')[['mia_auc','loss_gap']].agg(['mean','std']).round(4).to_string())

In [ ]:
# Ve histogram AUC
fig, ax = plt.subplots(figsize=(8,4.5))
configs = ['fedavg','fedprox','fedprox_dp']
colors = {'fedavg':'tab:blue','fedprox':'tab:orange','fedprox_dp':'tab:red'}
means = [df_mia[df_mia['config']==c]['mia_auc'].mean() for c in configs]
stds  = [df_mia[df_mia['config']==c]['mia_auc'].std()  for c in configs]
bars = ax.bar(configs, means, yerr=stds, capsize=8,
              color=[colors[c] for c in configs], edgecolor='black')
ax.axhline(0.5, color='gray', ls='--', label='Mức ngẫu nhiên (AUC=0.5)')
for b, m, s in zip(bars, means, stds):
    ax.text(b.get_x()+b.get_width()/2, m+ (s if s==s else 0)+0.01,
            f'{m:.3f}±{(s if s==s else 0):.3f}', ha='center', fontsize=9)
ax.set_ylabel('MIA AUC'); ax.set_ylim(0.4, 1.0)
ax.set_title('Membership Inference AUC theo cấu hình')
ax.grid(axis='y', alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig('runs/p3_mia.png', dpi=150)
plt.show()

## Xuất file cho báo cáo

Tải các file CSV/PNG để chèn vào LaTeX.

In [ ]:
from google.colab import files
for f in ['runs/p2_sweep.csv','runs/p2_sweep_curve.png',
          'runs/p3_mia.csv','runs/p3_mia.png']:
    if Path(f).exists():
        files.download(f)

# In bang tom tat cho copy vao LaTeX
print('\n=== BANG 1: Sweep σ - ε ===')
print(df_sweep.to_string(index=False))
print('\n=== BANG 2: MIA ===')
print(df_mia.groupby('config')[['mia_auc','mia_attack_acc_best','loss_gap']].agg(['mean','std']).round(4).to_string())
if 'df_auto' in globals():
    print('\n=== BANG 3: Adaptive clip vs C=1 ===')
    print(df_auto.groupby('config')[['test_acc','test_f1']].agg(['mean','std']).round(4).to_string())

## Diễn giải và giới hạn

- Bảng mean ± std chỉ xuất khi từng cấu hình có đủ số seed đã khai báo; std là độ lệch chuẩn mẫu (`ddof=1`). Nếu còn thiếu run, không điền số 0/NaN thay cho kết quả.
- Test chỉ được gọi sau khi chọn round bằng validation Macro-F1.
- DP ở đây vẫn là triển khai nghiên cứu: cần xác nhận sampling, RNG, clipping, accountant và threat model trước khi tuyên bố bảo đảm. Tăng batch size có thể giảm nhiễu trung bình trên gradient, nhưng làm thay đổi sampling rate và epsilon; luôn tính lại epsilon.
- `linear` giảm trainable parameters đáng kể, song chưa đảm bảo accuracy >80%. Kết quả mới phải được chạy và báo cáo trung thực.